In [ ]:
import torch
import cv2
import numpy as np

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("Is CUDA available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA version: 12.1
Is CUDA available?: True
GPU name: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
import pandas as pd 

path = "D:\BODYPAR\Graphonomy\Graphonomy-master\segmentation_summary.csv"
data = pd.read_csv(path)

In [10]:
import os
img_root = "D:/BODYPAR/Graphonomy/Graphonomy-master/img"
img_output = "D:/BODYPAR/Graphonomy/Graphonomy-master/img_par"
img_thermal = "D:/BODYPAR/Graphonomy/Graphonomy-master/img_thermal"
jpg_files = [f for f in os.listdir(img_root) if f.lower().endswith("flir.jpg")]
print(jpg_files) 

['P010_1_FLIR.jpg', 'P010_2_FLIR.jpg', 'P011_1_FLIR.jpg', 'P011_2_FLIR.jpg', 'P012_1_FLIR.jpg', 'P012_2_FLIR.jpg', 'P013_1_FLIR.jpg', 'P013_2_FLIR.jpg', 'P014_1_FLIR.jpg', 'P015_1_FLIR.jpg', 'P015_2_FLIR.jpg', 'P016_1_FLIR.jpg', 'P017_1_FLIR.jpg', 'P017_2_FLIR.jpg', 'P018_1_FLIR.jpg', 'P018_2_FLIR.jpg', 'P019_1_FLIR.jpg', 'P019_2_FLIR.jpg', 'P020_1_FLIR.jpg', 'P020_2_FLIR.jpg', 'P021_1_FLIR.jpg', 'P021_2_FLIR.jpg', 'P022_1_FLIR.jpg', 'P022_2_FLIR.jpg', 'P023_1_FLIR.jpg', 'P023_2_FLIR.jpg', 'P024_1_FLIR.jpg', 'P024_2_FLIR.jpg', 'P025_1_FLIR.jpg', 'P025_2_FLIR.jpg', 'P026_1_FLIR.jpg', 'P026_2_FLIR.jpg', 'P027_1_FLIR.jpg', 'P027_2_FLIR.jpg', 'P028_1_FLIR.jpg', 'P028_2_FLIR.jpg', 'P029_1_FLIR.jpg', 'P029_2_FLIR.jpg', 'P030_1_FLIR.jpg', 'P031_1_FLIR.jpg', 'P031_2_FLIR.jpg', 'P032_1_FLIR.jpg', 'P032_2_FLIR.jpg', 'P033_1_FLIR.jpg', 'P033_2_FLIR.jpg', 'P034_1_FLIR.jpg', 'P034_2_FLIR.jpg', 'P036_1_FLIR.jpg', 'P036_2_FLIR.jpg', 'P037_1_FLIR.jpg', 'P037_2_FLIR.jpg', 'P038_1_FLIR.jpg', 'P038_2_FLI

In [69]:
import os
import shutil

input_dir = img_thermal
output_dir = img_root

os.makedirs(output_dir, exist_ok=True)

for i, fname in enumerate(sorted(os.listdir(input_dir))):
    if not fname.lower().endswith("thermal.jpg"):
        continue
    

    ext = os.path.splitext(fname)[1]
    name = os.path.splitext(fname)[0][1:4]# .jpg
    name_index=os.path.splitext(fname)[0][5]
    num = int(name)
    if(num >= 254):
        new_name = f"P{name}_{name_index}_FLIR.jpg"
        src = os.path.join(input_dir, fname)
        dst = os.path.join(output_dir, new_name)
        shutil.copy2(src, dst)  # copy + 保留 metadata


In [77]:
from ultralytics import YOLO
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# YOLOv8-seg
model = YOLO("yolov8n-seg.pt")

def GetRoughtOutlineYolo(img_path,output_path = "thermal_contour_only.jpg"):
    results = model(img_path)
    if results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        orig_img = cv2.imread(img_path)
        h, w = orig_img.shape[:2]
        if masks is not None and len(masks) > 0:
        
            contour_canvas = np.zeros((h, w, 3), dtype=np.uint8)

            for i, mask in enumerate(masks):
                # resize 
                mask = cv2.resize(mask, (w, h))
                mask = (mask > 0.5).astype(np.uint8) * 255  
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(contour_canvas, contours, -1, (255, 255, 255), 2) 

            cv2.imwrite(output_path, contour_canvas)
    else:
        print('fail: ', img_path)

In [82]:
def GetRoughtOutlineYoloThermal(img_path,output_path = "thermal_contour_only.jpg"):
    conf = 0.05
    results = model(img_path, conf=conf)
   

    orig_img = cv2.imread(img_path)
    h, w = orig_img.shape[:2]


    if results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        contour_canvas = np.zeros((h, w, 3), dtype=np.uint8)

        for i, mask in enumerate(masks):
            
            mask = cv2.resize(mask, (w, h))
            mask = (mask > 0.5).astype(np.uint8) * 255

            
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(contour_canvas, contours, -1, (255, 255, 255), 2)

        cv2.imwrite(output_path, contour_canvas)

In [ ]:
image_path = r"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\img\\P010_2_FLIR.jpg"
outline_flir = r"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\tempboundary\\P010_2_contour.jpg"
GetRoughtOutlineYolo(img_path=image_path,output_path=outline_flir)


image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img\P010_2_FLIR.jpg: 480x640 1 person, 1 chair, 52.2ms
Speed: 3.8ms preprocess, 52.2ms inference, 3.9ms postprocess per image at shape (1, 3, 480, 640)


In [ ]:
img_matched = "D:\BODYPAR\Graphonomy\Graphonomy-master\img_matched"
img_output = "D:\BODYPAR\Graphonomy\Graphonomy-master\img_output_new"

In [11]:
def get_largest_component(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return mask
    cnt = max(cnts, key=cv2.contourArea)
    largest = np.zeros_like(mask)
    cv2.drawContours(largest, [cnt], -1, 255, thickness=-1)  
    return largest

def multi_scale_template_match(thermal_path, flir_path, scales=None, out_path="match_result.png"):
    
    thermal = cv2.imread(thermal_path, cv2.IMREAD_GRAYSCALE)
    flir = cv2.imread(flir_path, cv2.IMREAD_GRAYSCALE)

    
    _, thermal = cv2.threshold(thermal, 10, 255, cv2.THRESH_BINARY)
    _, flir = cv2.threshold(flir, 10, 255, cv2.THRESH_BINARY)

    #thermal = get_largest_component(thermal)
    #flir = get_largest_component(flir)

    if scales is None:
        scales = np.linspace(0.5, 1.5, 11)  #  [0.5x ~ 1.5x]

    best_val = -1
    best_scale = None
    best_loc = None
    best_template = None

    
    for s in scales:
        new_w = int(thermal.shape[1] * s)
        new_h = int(thermal.shape[0] * s)
        if new_w < 10 or new_h < 10: 
            continue

        template = cv2.resize(thermal, (new_w, new_h))
        res = cv2.matchTemplate(flir, template, cv2.TM_CCOEFF_NORMED)
        min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(res)

        if max_val > best_val:
            best_val = max_val
            best_scale = s
            best_loc = max_loc
            best_template = template.copy()

    
    top_left = best_loc
    h, w = best_template.shape
    bottom_right = (top_left[0] + w, top_left[1] + h)

    flir_bgr = cv2.cvtColor(flir, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(flir_bgr, top_left, bottom_right, (0,0,255), 2)  

    cv2.imwrite(out_path, flir_bgr)
    print(f"[OK] scale={best_scale:.2f}, score={best_val:.3f}, loc = {best_loc},{out_path}")

    return best_scale, best_val, best_loc

In [98]:
def ReduceOutline(img_path,output_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = np.where(gray > 0, 255, 0).astype(np.uint8)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt = max(contours, key=cv2.contourArea)

    out = np.zeros_like(img)
    cv2.drawContours(out, [cnt], -1, (0,255,0), 2)  

    cv2.imwrite(output_path, out)


In [ ]:
scale, score, loc = multi_scale_template_match(
    "D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\tempboundary\\P010_2_thermal_contour.jpg",
    "D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\tempboundary\\P010_2_contour_fix.jpg",
    scales=np.linspace(0.3, 2.0, 20)  
)

[OK] scale=1.55, score=0.255, loc = (70, 47),match_result.png


In [12]:
data_scale = []

for file in jpg_files:
    file_name = file[:-9]
    thermal_outline = f"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\tempboundary\\{file_name}_thermal_contour.jpg"
    flir_outline = f"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\tempboundary\\{file_name}_contour_fix.jpg"
    output_name = f"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\imgtemp\\{file_name}_macthed.jpg"
    
    scale, score, loc = multi_scale_template_match(
        thermal_path=thermal_outline,
        flir_path=flir_outline,
        out_path=output_name,
        scales=np.linspace(0.3, 2.0, 20)  
    )
    data_scaled = {'img_name' : file_name, 'scale' : scale,'score': score,'loc': loc}
    data_scale.append(data_scaled)

data_scale_df = pd.DataFrame(data_scale)
data_scale_df.head()

[OK] scale=1.55, score=0.415, loc = (71, 46),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P010_1_macthed.jpg
[OK] scale=1.55, score=0.255, loc = (70, 47),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P010_2_macthed.jpg
[OK] scale=0.30, score=0.226, loc = (305, 315),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P011_1_macthed.jpg
[OK] scale=0.30, score=0.245, loc = (399, 359),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P011_2_macthed.jpg
[OK] scale=0.30, score=0.244, loc = (361, 407),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P012_1_macthed.jpg
[OK] scale=0.30, score=0.299, loc = (226, 328),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P012_2_macthed.jpg
[OK] scale=1.55, score=0.260, loc = (71, 48),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P013_1_macthed.jpg
[OK] scale=1.64, score=0.291, loc = (49, 52),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P013_2_macthed.jpg
[OK] scale=0.30, score=0.278, loc = (336, 407),D:\BODYPAR\Graphonomy\Graphonomy-master\imgtemp\P

,img_name,scale,score,loc
0,P010_1,1.552632,0.414863,"(71, 46)"
1,P010_2,1.552632,0.254960,"(70, 47)"
2,P011_1,0.300000,0.225691,"(305, 315)"
3,P011_2,0.300000,0.245246,"(399, 359)"
4,P012_1,0.300000,0.243514,"(361, 407)"


In [13]:
fal_df = data_scale_df[data_scale_df['scale'] < 1]
fal_df

,img_name,scale,score,loc
2,P011_1,0.300000,0.225691,"(305, 315)"
3,P011_2,0.300000,0.245246,"(399, 359)"
4,P012_1,0.300000,0.243514,"(361, 407)"
5,P012_2,0.300000,0.298977,"(226, 328)"
8,P014_1,0.300000,0.278452,"(336, 407)"
...,...,...,...,...
478,P281_2,0.300000,0.176467,"(307, 355)"
479,P282_1,0.300000,0.253501,"(343, 289)"
480,P282_2,0.389474,0.222151,"(262, 302)"
481,P283_1,0.300000,0.191086,"(357, 252)"


In [129]:
from ultralytics import YOLO


model = YOLO("yolov8n-seg.pt")

def GetYOLOBox(img_path,output_path):
    results = model(img_path)

    results[0].save(output_path)
    result = results[0]
    boxes = result.boxes.xyxy.cpu().numpy()   
    confs = result.boxes.conf.cpu().numpy()   
    clss  = result.boxes.cls.cpu().numpy()    

    for (box, conf, cls) in zip(boxes, confs, clss):
        if(cls == 0):
            x1, y1, x2, y2 = map(int, box)
            w, h = x2 - x1, y2 - y1
            return x1,y1,w,h
        
    print(f" {img_path} No People Detection")
    return None

In [ ]:
output_box = r"D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\imgtemp\\failed"
erro_matched_files = fal_df['img_name'].unique().tolist()
for file in erro_matched_files:
    file_name = file
    thermal_img = img_thermal + f'\{file_name}_thermal.jpg'
    flir_img = img_root + f'\{file_name}_FLIR.jpg'
    thermal_box = output_box  + f'\{file_name}_thermal_box.jpg'
    flir_box = output_box  + f'\{file_name}_FLIR_box.jpg'
    
    box_thermal = GetYOLOBox(thermal_img,thermal_box)
    box =  GetYOLOBox(flir_img,flir_box)


image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P011_1_thermal.jpg: 480x640 1 person, 1 traffic light, 1 vase, 52.1ms
Speed: 3.8ms preprocess, 52.1ms inference, 5.2ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img\P011_1_FLIR.jpg: 480x640 1 person, 52.3ms
Speed: 3.6ms preprocess, 52.3ms inference, 3.4ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P011_2_thermal.jpg: 480x640 1 person, 46.6ms
Speed: 3.9ms preprocess, 46.6ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img\P011_2_FLIR.jpg: 480x640 1 person, 45.2ms
Speed: 3.3ms preprocess, 45.2ms inference, 4.1ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P012_1_thermal.jpg: 480x640 1 tv, 1 vase, 45.4ms
Speed: 3.6ms preprocess, 45.4ms inference, 3.7ms postprocess per 

In [6]:
import cv2
import numpy as np

def ssim(img1, img2):

    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2

    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)

    mu1 = cv2.GaussianBlur(img1, (11, 11), 1.5)
    mu2 = cv2.GaussianBlur(img2, (11, 11), 1.5)

    mu1_sq = mu1 * mu1
    mu2_sq = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = cv2.GaussianBlur(img1 * img1, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(img2 * img2, (11, 11), 1.5) - mu2_sq
    sigma12   = cv2.GaussianBlur(img1 * img2, (11, 11), 1.5) - mu1_mu2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()


def align_by_scale(img_orig, img_therm, scales=None, shift_range=10, step=2, vis_path="VI.jpg"):
    #if scales is None:
    scales = np.linspace(1.2, 1.7, 11)

    # GRAY
    g_therm = cv2.cvtColor(img_therm, cv2.COLOR_BGR2GRAY)
    h_t, w_t = g_therm.shape

    best_score, best_scale, best_aligned = -1, None, None

    for s in scales:
        # WHOLE PICTURE
        scaled = cv2.resize(img_orig, (int(img_orig.shape[1]/s), int(img_orig.shape[0]/s)))

       
        H, W = scaled.shape[:2]
        if H < h_t or W < w_t:
            continue  
        cx, cy = W // 2, H // 2
       # HIFT (dx, dy)
        for dy in range(-shift_range, shift_range+1, step):
            for dx in range(-shift_range, shift_range+1, step):
                x1 = cx - w_t//2 + dx
                y1 = cy - h_t//2 + dy
                x2, y2 = x1 + w_t, y1 + h_t

                if x1 < 0 or y1 < 0 or x2 > W or y2 > H:
                    continue

                roi_scaled = scaled[y1:y2, x1:x2]

                g_scaled = cv2.cvtColor(roi_scaled, cv2.COLOR_BGR2GRAY)
                score = ssim(g_therm, g_scaled)

                if score > best_score:
                    best_score = score
                    best_scale = s
                    best_shift = (dx, dy)
                    best_aligned = roi_scaled

    
    if vis_path and best_aligned is not None:
        overlay = cv2.addWeighted(best_aligned, 0.5, img_therm, 0.5, 0)
        concat = np.hstack((img_therm, best_aligned, overlay))
        cv2.imwrite(vis_path, concat)
       
    return best_aligned, best_scale, best_score, best_shift

In [ ]:
erro_matched_files = fal_df['img_name'].unique().tolist()
#erro_matched_files = ['P011_1']
align_data = []
for file in erro_matched_files:
    file_name = file
    thermal_img = img_thermal + f'\{file_name}_thermal.jpg'
    flir_img = img_root + f'\{file_name}_FLIR.jpg'
    img_orig = cv2.imread(flir_img)
    img_therm = cv2.imread(thermal_img)
    img_output = f'D:\\BODYPAR\\Graphonomy\\Graphonomy-master\\imgtemp\\{file_name}_matched1.jpg' 
    aligned, scale, score,best_shift = align_by_scale(
        img_orig, img_therm, 
        scales=[1.55263158, 1.64210526, 1.46315789, 1.37368421],
        vis_path = img_output
    )
    data_scaled = {'img_name' : file_name, 'scale' : scale,'score': score,'loc': best_shift}
    align_data.append(data_scaled)
    print(f"best_scale={scale:.2f}, best_score={score:.3f},best_shift = {best_shift}")

best_scale=1.55, best_score=0.176,best_shift = (-6, 4)
best_scale=1.50, best_score=0.162,best_shift = (6, -6)
best_scale=1.55, best_score=0.200,best_shift = (-6, 4)
best_scale=1.55, best_score=0.159,best_shift = (0, -4)
best_scale=1.55, best_score=0.156,best_shift = (0, -4)
best_scale=1.55, best_score=0.239,best_shift = (0, -4)
best_scale=1.55, best_score=0.186,best_shift = (0, -4)
best_scale=1.55, best_score=0.178,best_shift = (-8, 4)
best_scale=1.55, best_score=0.198,best_shift = (-6, 4)
best_scale=1.55, best_score=0.238,best_shift = (-2, -4)
best_scale=1.55, best_score=0.240,best_shift = (0, -4)
best_scale=1.55, best_score=0.160,best_shift = (-2, -4)
best_scale=1.55, best_score=0.140,best_shift = (-8, 4)
best_scale=1.55, best_score=0.138,best_shift = (-8, 4)
best_scale=1.55, best_score=0.216,best_shift = (-6, 4)
best_scale=1.60, best_score=0.218,best_shift = (-6, 4)
best_scale=1.55, best_score=0.159,best_shift = (-2, -4)
best_scale=1.50, best_score=0.177,best_shift = (6, -6)
best_sc

In [ ]:
align_df = pd.DataFrame(align_data)

In [ ]:
data_scale_df = data_scale_df.merge(align_df, on='img_name', how='left', suffixes=('', '_a'))
for c in ['scale', 'score', 'loc']:
    data_scale_df[c] = data_scale_df[c + '_a'].combine_first(data_scale_df[c])
    data_scale_df.drop(columns=c + '_a', inplace=True)

In [142]:
data_scale_df.to_csv('new_full_fixed.csv')

In [ ]:
def draw_by_scale(img_orig, img_therm, best_scale, best_shift,vis_path="VI.jpg"):
    
    img_orig = cv2.imread(img_orig)
    img_therm = cv2.imread(img_therm)
    # GRAY
    g_therm = cv2.cvtColor(img_therm, cv2.COLOR_BGR2GRAY)
    h_t, w_t = g_therm.shape

    scaled = cv2.resize(img_orig, (int(img_orig.shape[1]/best_scale), int(img_orig.shape[0]/best_scale)))
    H, W = scaled.shape[:2]
    cx, cy = W // 2, H // 2
    dx,dy = best_shift
    x1 = cx - w_t//2 + dx
    y1 = cy - h_t//2 + dy
    x2, y2 = x1 + w_t, y1 + h_t
    best_aligned = scaled[y1:y2, x1:x2]
    
    overlay = cv2.addWeighted(best_aligned, 0.5, img_therm, 0.5, 0)
    concat = np.hstack((img_therm, best_aligned, overlay))
    cv2.imwrite(vis_path, concat)
       


In [151]:
img_orig = "D:\BODYPAR\Graphonomy\Graphonomy-master\img\P011_1_FLIR.jpg"
img_therm = "D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P011_1_thermal.jpg"
#1.55	0.176244	(-6, 4)
draw_by_scale(img_orig=img_orig,img_therm=img_therm,best_scale=1.55,best_shift=(-6,4))

In [ ]:
# apply outline to compare and match

In [ ]:
# to check the match
img_matchVis = r"D:\BODYPAR\Graphonomy\Graphonomy-master\img_matchvis"
def VisMatchResult(img_name, thermal_img,matched_img):
    
    loc = data_scale_df.loc[data_scale_df['img_name'] == img_name, 'loc'].values[0]
    x,y = loc
    scale = data_scale_df.loc[data_scale_df['img_name'] == img_name, 'scale'].values[0]

    outline = cv2.imread(matched_img)
    thermal = cv2.imread(thermal_img)
    h,w = thermal.shape[:2]
    h = int(h * scale)
    w = int(w * scale)
    x, y = int(x), int(y)
    thermal_resized = cv2.resize(thermal, (int(w ), int(h)))

    overlay = outline.copy()
    overlay[y:y+h, x:x+w] = thermal_resized

    alpha = 0.5
    roi = outline[y:y+h, x:x+w]
    blended = cv2.addWeighted(roi, 1-alpha, thermal_resized, alpha, 0)
    overlay[y:y+h, x:x+w] = blended
    vis_img = img_matchVis + f'\{img_name}' + 'matched.jpg'
    cv2.imwrite(vis_img, overlay)

In [ ]:
import ast
def crop_from_template_match(
    flir_path,
    thermal_path,
    best_scale,
    best_loc,
    out_path="flir_crop.png"
):
    flir = cv2.imread(flir_path)
    thermal = cv2.imread(thermal_path)

    Ht, Wt = thermal.shape[:2]
    if isinstance(best_loc, str):
        best_loc = ast.literal_eval(best_loc)


    x, y = map(int, best_loc)

    # scaled thermal 
    w = int(Wt * best_scale)
    h = int(Ht * best_scale)

    Hf, Wf = flir.shape[:2]
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(Wf, x + w)
    y2 = min(Hf, y + h)

    if x1 >= x2 or y1 >= y2:
        raise ValueError("Empty crop from template match")

    roi = flir[y1:y2, x1:x2]

    # resize
    roi_resized = cv2.resize(roi, (Wt, Ht))

    cv2.imwrite(out_path, roi_resized)
  




In [ ]:
import cv2
import numpy as np

def crop_by_scale_and_shift(
    img_orig_path,
    img_therm_path,
    best_scale,
    best_shift,
    out_path="flir_crop_resized.png"
):
    img_orig = cv2.imread(img_orig_path)
    img_therm = cv2.imread(img_therm_path)

    h_t, w_t = img_therm.shape[:2]

    # resize
    scaled = cv2.resize(
        img_orig,
        (
            int(img_orig.shape[1] / best_scale),
            int(img_orig.shape[0] / best_scale)
        )
    )

    H, W = scaled.shape[:2]
    cx, cy = W // 2, H // 2

    if isinstance(best_shift, str):
        best_shift = ast.literal_eval(best_shift)
    dx, dy = map(int, best_shift)
    #crop box
    x1 = cx - w_t // 2 + dx
    y1 = cy - h_t // 2 + dy
    x2 = x1 + w_t
    y2 = y1 + h_t
    sx1 = max(0, x1)
    sy1 = max(0, y1)
    sx2 = min(W, x2)
    sy2 = min(H, y2)

    if sx1 >= sx2 or sy1 >= sy2:
        raise ValueError(f"Empty crop: shift={best_shift}, scale={best_scale}")

    tx1 = sx1 - x1
    ty1 = sy1 - y1
    tx2 = tx1 + (sx2 - sx1)
    ty2 = ty1 + (sy2 - sy1)

    
    cropped = np.zeros_like(img_therm)
    cropped[ty1:ty2, tx1:tx2] = scaled[sy1:sy2, sx1:sx2]

    cv2.imwrite(out_path, cropped)
    #return cropped


In [ ]:
img_orig = "D:\BODYPAR\Graphonomy\Graphonomy-master\img\P011_1_FLIR.jpg"
img_therm = "D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P011_1_thermal.jpg"
#1.55	0.176244	(-6, 4)

crop_by_scale_and_shift(
    img_orig,
    img_therm,
    1.55,
    (-6, 4),
    out_path="flir_crop_resized.png"
)

In [167]:
img_orig = "D:\BODYPAR\Graphonomy\Graphonomy-master\img\P010_1_FLIR.jpg"
img_therm = "D:\BODYPAR\Graphonomy\Graphonomy-master\img_thermal\P010_1_thermal.jpg"
#1.55	0.176244	(-6, 4)

crop_from_template_match(
    flir_path = img_orig,
    thermal_path= img_therm,
    best_scale = 1.552632,
    best_loc = (71, 46),
    out_path="flir_crop_resized.png"
)

In [33]:
img_crop = r"D:\BODYPAR\Graphonomy\Graphonomy-master\img_crop_par"
img_par = r"D:\BODYPAR\Graphonomy\Graphonomy-master\img_par"
data_scale = []
for image_name in data_scale_df['img_name'].tolist():
    #P010_1_FLIR_parsed.png
    image_name_flir = img_par + f'/{image_name}' +  '_FLIR_parsed_gray.png'
    image_name_thermal = f'{img_thermal}/' +  image_name + '_thermal.jpg'
    
    loc = data_scale_df.loc[data_scale_df['img_name'] == image_name, 'loc'].values[0]
    
    scale = data_scale_df.loc[data_scale_df['img_name'] == image_name, 'scale'].values[0]
    cropped_path = img_crop + f'\{image_name}_par_croped_gray.jpg'
    
    if image_name in fal_df['img_name'].unique().tolist():
        crop_by_scale_and_shift(
            image_name_flir,
            image_name_thermal,
            scale,
            loc,
            out_path=cropped_path
        )
    else:
        crop_from_template_match(
            flir_path = image_name_flir,
            thermal_path= image_name_thermal,
            best_scale =scale,
            best_loc = loc,
            out_path=cropped_path
        )

In [ ]:
# visualize the alignment

def visualize_crop_on_thermal(
    thermal_path,
    crop_path,
    out_path="vis.png",
    alpha=0.5
):
    thermal = cv2.imread(thermal_path)
    crop = cv2.imread(crop_path)

    if thermal is None or crop is None:
        raise ValueError("Failed to load images")

    if thermal.shape != crop.shape:
        raise ValueError(
            f"Shape mismatch: thermal {thermal.shape}, crop {crop.shape}"
        )

    # overlay
    overlay = cv2.addWeighted(
        thermal, 1 - alpha,
        crop, alpha,
        0
    )

    # concat
    vis = np.hstack([thermal, overlay, crop])

    cv2.imwrite(out_path, vis)
   

In [171]:
img_crop = r"D:\BODYPAR\Graphonomy\Graphonomy-master\img_crop1"
img_vi = r"D:\BODYPAR\Graphonomy\Graphonomy-master\match_temp"
data_scale = []
for image_name in data_scale_df['img_name'].tolist():
    image_name_crop = img_crop+ f'/{image_name}' +  '_croped.jpg'
    image_name_thermal = f'{img_thermal}/' +  image_name + '_thermal.jpg'
    image_name_vi = f'{img_vi}/' +  image_name + '_vi.jpg'
    visualize_crop_on_thermal(
        image_name_thermal,
        image_name_crop,
        out_path=image_name_vi,
        alpha=0.5
    )
    

In [5]:
data_scale_df.head()

,Unnamed: 0,img_name,scale,score,loc
0,0,P010_1,1.552632,0.414863,"(71, 46)"
1,1,P010_2,1.552632,0.254960,"(70, 47)"
2,2,P011_1,1.550000,0.176244,"(-6, 4)"
3,3,P011_2,1.500000,0.162467,"(6, -6)"
4,4,P012_1,1.550000,0.200189,"(-6, 4)"


In [15]:
fal_df.to_csv('fal_match.csv')

In [ ]:
img_crop = r"D:\BODYPAR\Graphonomy\Graphonomy-master\img_crop1_par"
data_scale = []
for image_name in data_scale_df['img_name'].tolist():
    image_name_flir = img_root+ f'/{image_name}' +  '_FLIR.jpg'
    image_name_thermal = f'{img_thermal}/' +  image_name + '_thermal.jpg'
    
    loc = data_scale_df.loc[data_scale_df['img_name'] == image_name, 'loc'].values[0]
    
    scale = data_scale_df.loc[data_scale_df['img_name'] == image_name, 'scale'].values[0]
    cropped_path = img_crop + f'\{image_name}_croped.jpg'
    
    if image_name in fal_df['img_name'].unique().tolist():
        # if the first alignment can't work -> change to the new alignment
        crop_by_scale_and_shift(
            image_name_flir,
            image_name_thermal,
            scale,
            loc,
            out_path=cropped_path
        )
    else:
        crop_from_template_match(
            flir_path = image_name_flir,
            thermal_path= image_name_thermal,
            best_scale =scale,
            best_loc = loc,
            out_path=cropped_path
        )

In [ ]:
# match body & color

In [15]:
import easyocr
reader = easyocr.Reader(['en'], gpu=False)
def ReadColorBar(path):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    colorbar = img_rgb[:, w-50:w, :]
    top_region = colorbar[0:50, :] 
    bottom_region = colorbar[-50:, :]
    top_result = reader.readtext(top_region)[0][1]
    bottom_result = reader.readtext(bottom_region)[0][1]
    return {'TopTemp':top_result,'BottomTemp':bottom_result}



Using CPU. Note: This module is much faster with a GPU.


In [ ]:
import easyocr
reader = easyocr.Reader(['en'], gpu=False)
def safe_read_text(reader, img):
    try:
        res = reader.readtext(img)
        if len(res) == 0:
            return None
        return res[0][1]
    except Exception as e:
        print("OCR failed:", e)
        return None


def ReadColorBar(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError("Image not found")

    h, w = img.shape[:2]

    # 色标在右侧
    colorbar = img[:, int(w*0.9):w]

    top_region = colorbar[0:50, :]
    bottom_region = colorbar[-50:, :]

    top_text = safe_read_text(reader, top_region)
    bottom_text = safe_read_text(reader, bottom_region)

    return {
        "TopTemp": top_text,
        "BottomTemp": bottom_text
    }


Using CPU. Note: This module is much faster with a GPU.


In [ ]:
import cv2
import numpy as np

def GetImageThermalByBodyPartFix(
    image_name,
    thermalImgPath,
    grayMaskPath,
    temp_max,
    temp_min
):
   

    try:
       
        thermal = cv2.imread(thermalImgPath)
        mask = cv2.imread(grayMaskPath)

        if thermal is None:
            print("Thermal image not found:", thermalImgPath)
            return None
        if mask is None:
            print("Mask image not found:", grayMaskPath)
            return None

        thermal = cv2.cvtColor(thermal, cv2.COLOR_BGR2RGB)
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        h, w = mask.shape

        
        # Map temp -> Color

        thermal_gray = cv2.cvtColor(thermal, cv2.COLOR_RGB2GRAY)

        thermal_norm = thermal_gray / 255.0
        temp_map = temp_min + thermal_norm * (temp_max - temp_min)

    
        label_map = {
            1: "hat",
            2: "hair",
            4: "sunglasses",
            5: "upper_clothes",
            6: "dress",
            7: "coat",
            8: "socks",
            9: "pants",
            10: "jumpsuits",
            11: "scarf",
            12: "skirt",
            13: "face",
            14: "left_arm",
            15: "right_arm",
            16: "left_leg",
            17: "right_leg",
            18: "left_shoe",
            19: "right_shoe",
            20:"torso_skin"
        }

        result = {"image": image_name}

        # calculate the body part temperature
        for label_val, part_name in label_map.items():

            region_mask = (mask == label_val)

            if np.sum(region_mask) == 0:
                result[f"{part_name}_mean"] = np.nan
                result[f"{part_name}_max"] = np.nan
                result[f"{part_name}_min"] = np.nan
                
                result[f"{part_name}_p90"] = np.nan
                result[f"{part_name}_p10"] = np.nan
                result[f"{part_name}_std"] = np.nan
                continue

            temps = temp_map[region_mask]

            result[f"{part_name}_mean"] = float(np.mean(temps))
            result[f"{part_name}_max"] = float(np.max(temps))
            result[f"{part_name}_min"] = float(np.min(temps))
           
            result[f"{part_name}_p90"] = float(np.percentile(temps, 90))
            result[f"{part_name}_p10"] = float(np.percentile(temps, 10))
            result[f"{part_name}_std"] = float(np.std(temps, ddof=1))

    
        return result

    except Exception as e:
        print("Error in:", image_name)
        print(e)
        return None

In [ ]:
rows = []
output_df_el = pd.read_csv('bobody_part.csv')
for row in output_df_el.itertuples():
    image = row.image
    
    row = GetImageThermalByBodyPartFix(
        image,
        f"img_thermal/{image}_thermal.jpg",
        f"img_crop_par/{image}_par_croped_gray.jpg",
        temp_max = row.colorbar_max,
        temp_min = row.colorbar_min
    )

    if row is not None:
        rows.append(row)
    else:
        print("Skipped:", image)

temp_df = pd.DataFrame(rows)

d:\conda_envs\graphonomy_thermal\lib\site-packages\numpy\core\_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
d:\conda_envs\graphonomy_thermal\lib\site-packages\numpy\core\_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
